In [ ]:
!pip install -q -U bitsandbytes peft transformers accelerate feedparser beautifulsoup4 requests

In [1]:
import torch
import os
import feedparser
import requests
from bs4 import BeautifulSoup
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from IPython.display import display, clear_output

VERSION = "v2_2"
print(f"Inizializzazione RAG Comedian per: {VERSION}")

# 1. Trova l'adattatore
adapter_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files and VERSION in root:
        adapter_path = root
        break

if not adapter_path:
    raise FileNotFoundError(f"Adattatore {VERSION} non trovato!")

# 2. Configurazione 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 3. Carica Base Model
print(" Caricamento Zephyr Base...")
BASE_MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Carica Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 5. Innesta Adattatore
print(" Aggancio la personalità DPO...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("MODELLO PRONTO! Passiamo al Retrieval delle notizie!")

Inizializzazione RAG Comedian per: v2_2
 Caricamento Zephyr Base...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

 Aggancio la personalità DPO...
MODELLO PRONTO! Passiamo al Retrieval delle notizie!


In [3]:
import feedparser
import urllib.parse

def get_news_context(topic):
    """Cerca le ultime notizie su Google News relative al topic richiesto."""
    # Codifica il topic per l'URL (es: "Sanremo 2026" diventa "Sanremo%202026")
    query = urllib.parse.quote(topic)
    
    # URL di Google News RSS (in inglese per far felice il modello, ma puoi cercare topic italiani)
    url = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
    feed = feedparser.parse(url)
    
    if not feed.entries:
        return None
        
    # Estraiamo i primi 2 titoli più recenti per dare contesto al modello
    context = ""
    for i, entry in enumerate(feed.entries[:2]):
        context += f"- News Headline: {entry.title}\n"
        
    return context

print(" Motore di Retrieval inizializzato!")

 Motore di Retrieval inizializzato!


In [9]:
import random
import re  # <--- Il nostro spazzino magico

# Ora accetta due parametri: il prompt originale (per la chat) e il topic (per la ricerca)
def generate_rag_joke(user_prompt, search_topic, temperature=0.7): 
    
    # 1. RETRIEVAL (Ora è 100% silenzioso, niente print!)
    news_context = get_news_context(search_topic)
    
    context_prompt = ""
    if news_context:
        # Prompt reso più naturale per evitare l'allucinazione "(No news...)"
        context_prompt = (
            f"\n\nLATEST NEWS CONTEXT:\n{news_context}\n"
            f"Your joke MUST reference or twist these specific news events."
        )
    
    # 2. PERSONALITÀ (I tuoi prompt originali)
    style_dad_jokes = (
        "You are a professional comedy writer specializing in one-liners. "
        "Your goal is to make the user laugh with a SINGLE, sharp punchline. "
        "Do not ramble. Do not offer multiple options."
    )
    style_comedian = (
        "You are a witty stand-up comedian with a cynical style. "
        "Make ONE sarcastic observation or tell ONE mini-story about the topic. "
        "Do not use the Q&A format."
    )
    style_observational = (
        "You are a cynical observational stand-up comedian. "
        "Make a sharp, dark observation about the topic. "
        "Avoid the classic Q&A setup."
    )
    style_satire = (
        "You are a biting satirical writer. "
        "Deliver ONE sarcastic remark or dark humor statement about the user's topic. "
        "Be direct and end with a clever twist."
    )
    style_classic = (
        "You are a classic comedy writer specializing in punchy jokes. "
        "Tell a short, clever joke about the topic. You can use a classic Question & Answer setup (like 'Why did...') if it hits hard."
    )
    
    critical_rules = (
        "\n\nCRITICAL OUTPUT RULES:\n"
        "1. Output ONLY the raw joke text.\n"
        "2. NEVER use meta-labels (e.g., 'Cynical Comedian:', 'Mini-story time:', 'Here is a joke:').\n"
        "3. NEVER use conversational filler (e.g., 'Sure thing!', 'Certainly!').\n"
        "4. NEVER output system messages like '(No news headline provided)'."
    )

    # 3. SELEZIONE VERSIONE
    if (VERSION == "v2_1"):
        base_persona = random.choices([style_dad_jokes, style_comedian], weights=[50, 50], k=1)[0]
    elif (VERSION == "v2_2"):
        base_persona = random.choices([style_observational, style_satire, style_classic], weights=[40, 40, 20], k=1)[0]
        
    current_system_prompt = base_persona + critical_rules + context_prompt

    # 4. GENERAZIONE
    messages = [
        {"role": "system", "content": current_system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=150, do_sample=True, temperature=temperature, 
            top_k=40, top_p=0.90, repetition_penalty=1.15, pad_token_id=tokenizer.eos_token_id
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded.split("<|assistant|>")[-1].strip()
    
    # TAGLIOLA MIGLIORATA
    stop_phrases = ["Or how about", "Here's another", "Alternatively", "(Ba-dum-tish)", "(Boom tsh)", "(No news headline provided)"]
    for phrase in stop_phrases:
        if phrase in response:
            response = response.split(phrase)[0].strip()
            
    if "\n\n" in response:
        parts = response.split("\n\n")
        if any(word in parts[0].lower() for word in ["sure", "here", "certainly", "of course"]):
            if len(parts) > 1: response = parts[1].strip()
        else: response = parts[0].strip()

    preambles = ["Sure thing! ", "Here it is: ", "Sure, ", "Here is one: ", "Certainly! "]
    for p in preambles:
        if response.startswith(p): response = response[len(p):].strip()

    # --- SPAZZINO DEGLI HASHTAG ---
    # Rimuove qualsiasi parola che inizia con '#' e pulisce gli spazi doppi rimasti
    response = re.sub(r'#\w+', '', response)
    response = re.sub(r'\s+', ' ', response).strip()

    # Fallback super sicuro
    if not response:
        response = "I tried to make a joke about the news, but reality is already too depressing."

    return response

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_input = widgets.Text(
    placeholder='Scrivi un prompt (es: Tell me a joke about Elon Musk)...',
    layout=widgets.Layout(width='70%')
)

submit_button = widgets.Button(
    description="Invia", 
    button_style="primary", 
    icon='paper-plane'
)

output_area = widgets.Output(layout={
    'border': '1px solid #444', 
    'padding': '15px', 
    'margin': '10px 0'
})

def on_submit(_):
    # Salviamo il prompt esatto che hai digitato tu
    original_prompt = text_input.value.strip()
    if not original_prompt: return
    text_input.value = "" 
    
    # Estraiamo SOLO IL TOPIC per fare la ricerca di nascosto
    search_topic = original_prompt
    prefixes = ["tell me a joke about ", "make a joke about ", "make a sarcastic comment about ", "tell me a dark humor joke about ", "fai una battuta su ", "fammi una battuta su "]
    for p in prefixes:
        if search_topic.lower().startswith(p):
            search_topic = search_topic[len(p):].strip()
            break # Si ferma appena trova il prefisso
            
    with output_area:
        clear_output()
        # Mostriamo il TUO prompt intatto
        print(f" TU: {original_prompt}")
        print(" Zephyr sta leggendo le ultime notizie e scrivendo la battuta...")
        
        try:
            # Passiamo SIA il prompt per il LLM, SIA il topic per Google News
            response = generate_rag_joke(original_prompt, search_topic, temperature=0.7)
            
            clear_output()
            print(f" TU: {original_prompt}")
            print(f" ZEPHYR COMEDIAN ({VERSION} + RAG): {response}")
            
        except Exception as e:
            print(f" Errore: {e}")

submit_button.on_click(on_submit)

# Sostituiamo il vecchio on_submit con observe per eliminare il DeprecationWarning
def handle_enter(change):
    if change['new'] == text_input.value and text_input.value != '':
        on_submit(None)

text_input.continuous_update = False
text_input.observe(handle_enter, names='value')

print(f" Il palcoscenico è pronto per Zephyr DPO {VERSION}!")
display(widgets.HBox([text_input, submit_button]), output_area)

 Il palcoscenico è pronto per Zephyr DPO v2_2!


/tmp/ipykernel_116/67064117.py:55: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_submit)


Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…